# Iterative deepening & bidirectional search — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
DIRS=[(1,0),(-1,0),(0,1),(0,-1)]
def neighbors(p, shape=(5,5), walls=set()):
    for dr,dc in DIRS:
        q=(p[0]+dr,p[1]+dc)
        if 0<=q[0]<shape[0] and 0<=q[1]<shape[1] and q not in walls: yield q
def reconstruct(parent, goal):
    path=[]; x=goal
    while x is not None: path.append(x); x=parent.get(x)
    return path[::-1]
def draw(vals, title):
    plt.figure(figsize=(5,3)); plt.plot(vals,'-o'); plt.title(title)
def draw_grid(path=(), explored=(), frontier=(), walls=set(), start=(0,0), goal=(4,4), shape=(5,5), title='grid'):
    img=np.zeros(shape)
    for r,c in walls: img[r,c]=-1
    for r,c in explored: img[r,c]=.35
    for r,c in frontier: img[r,c]=.65
    for r,c in path: img[r,c]=1
    plt.figure(figsize=(4,4)); plt.imshow(img,cmap='viridis',vmin=-1,vmax=1); plt.scatter([start[1]],[start[0]],c='w',edgecolor='k'); plt.scatter([goal[1]],[goal[0]],c='r',edgecolor='k'); plt.title(title); plt.xticks(range(shape[1])); plt.yticks(range(shape[0])); plt.grid(color='w',alpha=.3)
print('setup ok')

## Depth and direction as memory-saving choices
IDDFS repeats cheap depth-limited passes; bidirectional search replaces one deep wave with two shallower waves.

In [ ]:
s=(0,0); g=(0,4); limit=3; seen=[]
def dls(u,d,par):
 seen.append(u)
 if u==g: return par
 if d==limit: return None
 for v in neighbors(u):
  if v not in par:
   par[v]=u; ans=dls(v,d+1,par)
   if ans: return ans
ans=dls(s,0,{s:None}); draw_grid((),seen,goal=g,title='limit 3 cuts off')
assert ans is None and g not in seen

In [ ]:
s=(0,0); g=(0,4); attempts=[]
def limited(L):
 par={s:None}; touched=[]
 def rec(u,d):
  touched.append(u)
  if u==g: return True
  if d==L: return False
  for v in neighbors(u):
   if v not in par:
    par[v]=u
    if rec(v,d+1): return True
  return False
 return rec(s,0),par,touched
for lim in range(6):
 ok,par,touched=limited(lim); attempts.append(len(touched))
 if ok: break
p=reconstruct(par,g); draw_grid(p,touched,goal=g,title=f'IDDFS finds depth {lim}')
assert lim==4 and len(p)-1==4

In [ ]:
plt.figure(figsize=(5,3)); plt.bar(range(len(attempts)),attempts); plt.xlabel('limit'); plt.ylabel('nodes touched'); plt.title(f'total touches {sum(attempts)}')
assert sum(attempts)==33 and attempts[-1]==15

In [ ]:
s=(0,0); g=(4,4); F={s}; B={g}; pf={s:None}; pb={g:None}; meet=None
while meet is None:
 NF=set()
 for u in F:
  for v in neighbors(u):
   if v not in pf: pf[v]=u; NF.add(v)
   if v in B: meet=v; break
  if meet: break
 F=NF
 if meet: break
 NB=set()
 for u in B:
  for v in neighbors(u):
   if v not in pb: pb[v]=u; NB.add(v)
   if v in F: meet=v; break
  if meet: break
 B=NB
front=reconstruct(pf,meet); back=[]; x=meet
while x is not None: back.append(x); x=pb.get(x)
p=front+back[1:]; draw_grid(p,set(pf)|set(pb),F|B,title=f'meet {meet}')
assert len(p)-1==8 and meet in pf and meet in pb

In [ ]:
b=4; d=8; bfs=b**d; bidir=2*b**(d//2); plt.figure(figsize=(5,3)); plt.bar(['one frontier','two frontiers'],[bfs,bidir]); plt.yscale('log'); plt.title('branching-factor arithmetic')
assert bfs==65536 and bidir==512 and bidir<bfs